# Keyword biasing on clinical speech: Granite Speech 5.0 vs Whisper on PriMock57 and AfriSpeech

**Question:** on real clinical speech that none of these models was trained on, does Granite Speech 5.0's keyword biasing get medical terms right more often than plain transcription and Whisper's prompting, and how often does it write listed terms that nobody said?

**Setup**
- **PriMock57** (CC BY 4.0): 57 mock GP consultations, doctor and patient on separate channels, with hand-timed utterance transcripts. Conversational speech with fillers and over-talk.
- **AfriSpeech-200 clinical test set** (CC BY-NC-SA 4.0, gated): read clinical sentences from many African accents, about 30 clips per accent. Quiet read speech, so it tests accents rather than acoustics.
- Neither dataset is in Granite Speech 5.0's published training data, and both postdate Whisper's.
- **Keyword lists:** each consultation (PriMock57) or block of clips (AfriSpeech) gets one list: its own rare terms (wordfreq Zipf below 3.0, such as *fexofenadine* or *co-codamol*) plus an equal number of **distractors**, rare terms from elsewhere in the dataset that nobody in that group said.
- **Conditions:** within each model, only the keyword list changes.

| Condition | Model | Keywords |
|---|---|---|
| A | Whisper-small | none |
| B | Whisper-small | the list as a Whisper prompt (`prompt_ids`) |
| C | Granite Speech 5.0 Turbo CTC (470M) | none (a CTC model takes no prompt) |
| D | Granite Speech 5.0 2B, `transcribe the speech to text.` | none |
| E | Granite Speech 5.0 2B, `transcribe the speech to text. Keywords: …` | the list |

- **Metrics:** WER; WER on listed words (B-WER) and on all other words (U-WER); target-term recall; distractor insertions per 100 utterances; speed (RTFx) and peak GPU memory. Confidence intervals come from a bootstrap that resamples whole consultations or speakers.

**TL;DR:** TODO(you) after running.

## Why this matters

[Granite Speech 5.0](https://huggingface.co/blog/ibm-granite/granite-speech-5) (IBM, 20–21 October 2026) is a 2B speech-language model that transcribes six languages and accepts up to 2,000 **bias keywords** in its prompt; the same release made the 470M Turbo CTC model multilingual and faster. IBM reports keyword-biasing F1 and Open ASR Leaderboard WER, but AMI and Earnings-22, the natural test sets for meetings and jargon, are both in its training data.

Clinical transcription is where keyword lists matter most: a patient's medication list is known before the visit, and a misheard drug name is worse than a misheard filler word. Biasing has a known failure mode too: the model writes a listed term because it was listed, not because it was said. This notebook measures both sides on two clinical datasets the models haven't seen, against Whisper's usual prompt-based biasing.

## Setup

Runs on a Colab T4. AfriSpeech-200 is gated: request access at [huggingface.co/datasets/intronhealth/afrispeech-200](https://huggingface.co/datasets/intronhealth/afrispeech-200), then add a Hugging Face token as a Colab secret named `HF_TOKEN`. Without it, the notebook runs on PriMock57 only.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

%pip install -q "transformers==5.20.2" "jiwer==4.0.0" "wordfreq==3.1.1" soundfile librosa

In [ ]:
import gc, json, random, re, tarfile, time, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import jiwer
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from huggingface_hub import HfApi, hf_hub_download
from tqdm.auto import tqdm
from transformers import (AutoModelForCTC, AutoModelForSpeechSeq2Seq, AutoProcessor,
                          WhisperForConditionalGeneration, WhisperProcessor)
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer
from wordfreq import zipf_frequency

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from importlib.metadata import version
print({p: version(p) for p in ["transformers", "torch", "jiwer", "wordfreq", "huggingface-hub", "librosa", "soundfile"]})
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| local")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found: choose Runtime -> Change runtime type -> T4 GPU. On a CPU this takes many hours.")

## Config

In [ ]:
SMOKE = False                     # True: 3 consultations and 2 accents, separate cache (about 15 minutes end to end)

# Models, pinned to the Hub revisions checked on 24 October 2026
WHISPER_ID, WHISPER_REV = "openai/whisper-small", "973afd24965f72e36ca33b3055d56a652f456b4d"
GRANITE_ID, GRANITE_REV = "ibm-granite/granite-speech-5.0-2b", "c075cf19e041a2e4d16c5ee24763e5b037d8f479"
CTC_ID, CTC_REV = "ibm-granite/granite-speech-5.0-470m-turboctc", "1dd23f46f6d4e6eae2531982ba2e33220086dd97"
GRANITE_DTYPE = "auto"            # "auto": bfloat16 on Ampere or newer, float16 on a T4. Set "float32" if float16 output looks garbled

# Prompts: within each model, conditions differ only by the keyword list
GRANITE_PROMPT = "transcribe the speech to text."
GRANITE_KW_PROMPT = "transcribe the speech to text. Keywords: {keywords}"   # IBM's keyword-biasing prompt
WHISPER_KW_PROMPT = "Keywords: {keywords}."                                   # passed to Whisper as prompt_ids

CONDITIONS = {                    # condition: (model, gets keywords, label)
    "A": ("whisper", False, "A Whisper-small"),
    "B": ("whisper", True, "B Whisper-small + keyword prompt"),
    "C": ("ctc", False, "C Granite Turbo CTC 470M"),
    "D": ("granite", False, "D Granite Speech 2B"),
    "E": ("granite", True, "E Granite Speech 2B + keywords"),
}
COMPARISONS = [("B", "A", "keywords, Whisper"), ("E", "D", "keywords, Granite 2B"),
               ("D", "A", "Granite 2B vs Whisper"), ("C", "A", "Turbo CTC vs Whisper"), ("E", "B", "both with keywords")]
BATCH = {"whisper": 16, "ctc": 16, "granite": 8}
MAX_NEW_TOKENS = 220              # the longest utterances are 30 s, about 100 words
WHISPER_MAX_PROMPT = 448 - MAX_NEW_TOKENS - 8   # Whisper's decoder holds 448 tokens in total

# Data
PRIMOCK_SHA = "cd2ac707ad03cb4d2531f4ec6b90c659bf4357c5"   # babylonhealth/primock57, last commit
PRIMOCK_N_CONSULTS = 3 if SMOKE else None                   # None = all 57
AFRI_REPO = "intronhealth/afrispeech-200"
AFRI_PER_ACCENT = 30              # clinical test clips per accent, from accents that have at least this many
AFRI_MAX_ACCENTS = 2 if SMOKE else 40
AFRI_GROUP = 5                    # clips that share one keyword list
MIN_SEC, MAX_SEC = 1.0, 30.0      # Whisper reads at most 30 s, so longer utterances are dropped for every model

# Keyword lists
RARE_ZIPF = 3.0                   # a word is a target term if its wordfreq Zipf frequency is below this (about 1 per million words)
MIN_LEN = 4                       # ...and it has at least this many letters
DISTRACTOR_RATIO = 1.0            # distractors per target term in each list
KW_MAX = 60                       # cap on list length (targets kept first)

N_BOOT = 2000
MIN_ACCENT_CLIPS = 30             # per-accent WER is shown for accents with at least this many clips

USE_DRIVE = True                  # keep transcripts on Google Drive so a disconnect resumes where it stopped
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/ML/2026-10-granite-speech-clinical-kwb")
else:
    if USE_DRIVE:
        print("Not on Colab: USE_DRIVE ignored, caching under outputs/")
    BASE_DIR = Path("outputs")
CACHE_DIR = BASE_DIR / ("cache_smoke" if SMOKE else "cache")
DATA_DIR, FIG_DIR = Path("data"), Path("figures")   # audio stays in the session (about 3 GB), not on Drive
for d in (CACHE_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("cache:", CACHE_DIR)

## Data

### Text normalisation and target terms

Every reference and transcript goes through Whisper's English normaliser before scoring: lower case, no punctuation, fillers such as "um" and "uh" removed, numbers and British spellings standardised. A **target term** is a normalised word of at least 4 letters with a wordfreq Zipf frequency below 3.0. On PriMock57 that picks out about 10–15 words per consultation, mostly drugs, conditions and symptoms.

In [ ]:
_spelling = json.load(open(hf_hub_download(WHISPER_ID, "normalizer.json", revision=WHISPER_REV)))
_normalizer = EnglishTextNormalizer(_spelling)


def norm(text):
    return re.sub(r"\s+", " ", _normalizer(text)).strip()


_zipf, _word_norm = {}, {}


def is_rare(tok):
    if len(tok) < MIN_LEN or not tok.isalpha():
        return False
    if tok not in _zipf:
        _zipf[tok] = zipf_frequency(tok, "en")
    return _zipf[tok] < RARE_ZIPF


WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]*[A-Za-z]")


def rare_terms(text):
    """Surface words in a reference whose normalised form contains a rare token: {lower: (surface, rare tokens)}."""
    out = {}
    for w in WORD_RE.findall(text):
        if w not in _word_norm:
            _word_norm[w] = tuple(t for t in norm(w).split() if is_rare(t))
        if _word_norm[w]:
            out.setdefault(w.lower(), (w, _word_norm[w]))
    return out


print(norm("Um, I've been taking co-codamol and Microgynon for my tumour."))
print(rare_terms("Um, I've been taking co-codamol and Microgynon for my tumour."))

### PriMock57

Audio and transcripts come from [babylonhealth/primock57](https://github.com/babylonhealth/primock57), pinned to its last commit. Each consultation has a doctor and a patient channel (16 kHz WAV) and a TextGrid of timed utterances. Utterances containing `<UNIN/>` (unintelligible) are dropped, `<UNSURE>` tags are removed but their text kept, and utterances under 1 s or over 30 s are dropped.

In [ ]:
PRIMOCK_RAW = f"https://raw.githubusercontent.com/babylonhealth/primock57/{PRIMOCK_SHA}"
PRIMOCK_MEDIA = f"https://media.githubusercontent.com/media/babylonhealth/primock57/{PRIMOCK_SHA}"   # audio is in Git LFS
INTERVAL_RE = re.compile(r'xmin = ([\d.]+)\s+xmax = ([\d.]+)\s+text = "((?:[^"]|"")*)"')


def fetch(url, path):
    path = Path(path)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".part")
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(path)
    return path


def read_textgrid(path):
    return [(float(a), float(b), t.replace('""', '"')) for a, b, t in INTERVAL_RE.findall(Path(path).read_text(encoding="utf-8"))]


def primock_utterances(consults, data_dir):
    rows, dropped = [], Counter()
    for cid in consults:
        for role in ("doctor", "patient"):
            tg = data_dir / f"{cid}_{role}.TextGrid"
            for i, (start, end, text) in enumerate(read_textgrid(tg)):
                if not text.strip():
                    continue
                if "<UNIN" in text:
                    dropped["unintelligible"] += 1
                    continue
                ref = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", text)).strip()
                if not norm(ref):
                    dropped["fillers only"] += 1
                    continue
                if not MIN_SEC <= end - start <= MAX_SEC:
                    dropped["under 1 s or over 30 s"] += 1
                    continue
                rows.append(dict(dataset="primock57", utt_id=f"{cid}_{role}_{i:03d}", group=cid, cluster=cid,
                                 subset=role, start=start, end=end, duration=end - start, ref=ref,
                                 audio=str(data_dir / f"{cid}_{role}.wav")))
    return pd.DataFrame(rows), dropped

In [ ]:
PRIMOCK_DIR = DATA_DIR / "primock57"
listing = PRIMOCK_DIR / "listing.json"
if not listing.exists():
    PRIMOCK_DIR.mkdir(parents=True, exist_ok=True)
    api_url = f"https://api.github.com/repos/babylonhealth/primock57/contents/transcripts?ref={PRIMOCK_SHA}"
    listing.write_text(json.dumps([f["name"] for f in json.load(urllib.request.urlopen(api_url))]))
consults = sorted({n.rsplit("_", 1)[0] for n in json.loads(listing.read_text()) if n.endswith(".TextGrid")})
if PRIMOCK_N_CONSULTS:
    consults = consults[:PRIMOCK_N_CONSULTS]

jobs = [(f"{PRIMOCK_RAW}/transcripts/{c}_{r}.TextGrid", PRIMOCK_DIR / f"{c}_{r}.TextGrid") for c in consults for r in ("doctor", "patient")]
jobs += [(f"{PRIMOCK_MEDIA}/audio/{c}_{r}.wav", PRIMOCK_DIR / f"{c}_{r}.wav") for c in consults for r in ("doctor", "patient")]
with ThreadPoolExecutor(8) as pool:
    list(tqdm(pool.map(lambda j: fetch(*j), jobs), total=len(jobs), desc="PriMock57 download"))
info = sf.info(str(PRIMOCK_DIR / f"{consults[0]}_doctor.wav"))
assert info.samplerate == 16000 and info.channels == 1, info

primock, primock_dropped = primock_utterances(consults, PRIMOCK_DIR)
print(f"{len(consults)} consultations, {len(primock)} utterances, {primock.duration.sum() / 3600:.2f} h of speech")
print("dropped:", dict(primock_dropped))
print(primock.groupby("subset").agg(utterances=("utt_id", "size"), hours=("duration", lambda s: s.sum() / 3600)))
primock.sample(3, random_state=SEED)[["utt_id", "duration", "ref"]]

### AfriSpeech-200 (clinical test set)

[AfriSpeech-200](https://huggingface.co/datasets/intronhealth/afrispeech-200) (Intron Health, CC BY-NC-SA 4.0) is gated, and its loading script doesn't run on current `datasets`, so this cell downloads `transcripts/test.csv` and only the audio archives it needs with `huggingface_hub`. It keeps clinical-domain clips of 1–30 s and takes 30 clips from each accent that has at least 30, up to 40 accents. In the public test CSV in Intron's [GitHub repo](https://github.com/intron-innovation/AfriSpeech-200) (transcripts only, no audio), 42 accents have at least 30 such clips, so this gives 1,200 clips. The column names were checked against that CSV; the cell still prints them, in case the Hub copy differs.

Licence: non-commercial, share-alike. Don't commit the audio; per-clip result files that include transcripts are marked CC BY-NC-SA in the README.

In [ ]:
def hf_token():
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("HF_TOKEN")
        except Exception:
            return None
    return os.environ.get("HF_TOKEN")


AFRI_COLS = {"path": ["audio_paths", "path", "audio_path", "audio_id", "file"], "ref": ["transcript", "text", "sentence"],
             "accent": ["accent"], "domain": ["domain"], "speaker": ["user_ids", "speaker_id", "user_id"],
             "duration": ["duration"]}


def pick_cols(df):
    found = {k: next((c for c in cands if c in df.columns), None) for k, cands in AFRI_COLS.items()}
    missing = [k for k in ("path", "ref", "accent", "domain") if found[k] is None]
    if missing:
        raise KeyError(f"AfriSpeech test.csv has no column for {missing}; columns are {list(df.columns)}. Edit AFRI_COLS.")
    return found


def clean_afri(text):
    text = re.sub(r"\s+([.,;:!?])", r"\1", str(text))   # 'hypertension .' -> 'hypertension.'
    return re.sub(r"(\d) s\b", r"\1s", text).strip()     # '50 s' -> '50s'


def load_afrispeech():
    token = hf_token()
    try:
        csv_path = hf_hub_download(AFRI_REPO, "transcripts/test.csv", repo_type="dataset", token=token)
        files = HfApi(token=token).list_repo_files(AFRI_REPO, repo_type="dataset")
    except Exception as e:
        print(f"AfriSpeech-200 skipped ({type(e).__name__}). Request access on its Hub page and add an HF_TOKEN Colab secret.")
        return pd.DataFrame()
    raw = pd.read_csv(csv_path)
    cols = pick_cols(raw)
    print("test.csv columns:", list(raw.columns), "\nusing:", cols)
    df = pd.DataFrame({k: raw[c] for k, c in cols.items() if c})
    df = df[df.domain.astype(str).str.lower().eq("clinical") & df.ref.notna()].copy()
    df["ref"] = df.ref.map(clean_afri)
    df = df[df.ref.map(norm).str.len() > 0]
    if "duration" in df:
        df = df[df.duration.between(MIN_SEC, MAX_SEC)]
    df["accent"] = df.accent.astype(str).str.strip().str.lower()
    counts = df.accent.value_counts()
    accents = list(counts[counts >= AFRI_PER_ACCENT].index[:AFRI_MAX_ACCENTS])
    df = df[df.accent.isin(accents)].groupby("accent").sample(AFRI_PER_ACCENT, random_state=SEED)

    afri_dir = DATA_DIR / "afrispeech"
    afri_dir.mkdir(parents=True, exist_ok=True)
    df["file"] = df.path.astype(str).map(lambda p: Path(p).name if p.endswith(".wav") else Path(p).name + ".wav")
    wanted = set(df.file) - {p.name for p in afri_dir.glob("*.wav")}
    for accent in tqdm(accents, desc="AfriSpeech archives"):
        folder = re.sub(r"[^a-z0-9]+", "-", accent).strip("-")   # 'ijaw(nembe)' -> 'ijaw-nembe', 'yoruba, hausa' -> 'yoruba-hausa'
        tars = [f for f in files if f.startswith(f"audio/{folder}/test/") and f.endswith(".tar.gz")]
        if not tars:
            print(f"  no test archive found for accent '{accent}'; its clips are dropped")
        for t in tars:
            if not wanted & set(df[df.accent == accent].file):
                break
            local = hf_hub_download(AFRI_REPO, t, repo_type="dataset", token=token)
            with tarfile.open(local) as tf:
                for m in tf.getmembers():
                    name = Path(m.name).name
                    if m.isfile() and name in wanted:
                        (afri_dir / name).write_bytes(tf.extractfile(m).read())
                        wanted.discard(name)
            Path(local).unlink(missing_ok=True)   # archives are large; the extracted clips are enough
    df["audio"] = df.file.map(lambda f: str(afri_dir / f))
    df = df[df.audio.map(lambda p: Path(p).exists())].copy()
    if "duration" not in df:
        df["duration"] = df.audio.map(lambda p: sf.info(p).duration)
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)   # mixed accents within each keyword group
    df["group"] = [f"afri_g{i // AFRI_GROUP:03d}" for i in range(len(df))]
    df["cluster"] = df.speaker.astype(str) if "speaker" in df else df.accent
    df["utt_id"] = df.file.str.removesuffix(".wav")
    df["dataset"], df["subset"] = "afrispeech", df.accent
    return df[["dataset", "utt_id", "group", "cluster", "subset", "duration", "ref", "audio"]]


afri = load_afrispeech()
if len(afri):
    print(f"{len(afri)} clips, {afri.subset.nunique()} accents, {afri.cluster.nunique()} speakers, {afri.duration.sum() / 3600:.2f} h")
    display(afri.sample(3, random_state=SEED)[["utt_id", "subset", "duration", "ref"]])

### Keyword lists

One list per group: a PriMock57 consultation (both channels) or a block of 5 AfriSpeech clips. A list holds the group's own target terms plus the same number of distractors, which are target terms from other groups whose words don't appear anywhere in this group's references. Lists are shuffled with a fixed seed so targets and distractors are mixed.

Every condition is scored against the same lists, including the ones that never see them (A, C and D), so the B-WER and insertion numbers are comparable.

In [ ]:
def build_keyword_lists(df):
    """{group: dict(keywords=[surface forms], targets=set of rare tokens, distractors=set of rare tokens)}"""
    terms = {g: {} for g in df.group.unique()}
    vocab = {g: set() for g in terms}
    for r in df.itertuples():
        terms[r.group].update(rare_terms(r.ref))
        vocab[r.group].update(norm(r.ref).split())
    pool = [(g, low, surf, toks) for g, ts in terms.items() for low, (surf, toks) in ts.items()]
    lists = {}
    for g in terms:
        rng = random.Random(f"{SEED}-{g}")
        targets = sorted(terms[g].values())[:KW_MAX]
        own = {low for low in terms[g]}
        cands = sorted({(low, surf, toks) for og, low, surf, toks in pool
                        if og != g and low not in own and not set(toks) & vocab[g]})
        n_dis = min(len(cands), round(len(targets) * DISTRACTOR_RATIO), KW_MAX - len(targets))
        dis = rng.sample(cands, n_dis)
        dis = list({low: (surf, toks) for low, surf, toks in dis}.values())
        kws = [s for s, _ in targets] + [s for s, _ in dis]
        rng.shuffle(kws)
        lists[g] = dict(keywords=kws, targets={t for _, ts in targets for t in ts},
                        distractors={t for _, ts in dis for t in ts})
    return lists


def describe_lists(name, lists):
    n_t = [len(v["targets"]) for v in lists.values()]
    n_d = [len(v["distractors"]) for v in lists.values()]
    print(f"{name}: {len(lists)} lists, median {np.median(n_t):.0f} target and {np.median(n_d):.0f} distractor tokens per list")

In [ ]:
DATA = {"primock57": primock}
if len(afri):
    DATA["afrispeech"] = afri
KW = {name: build_keyword_lists(df) for name, df in DATA.items()}
for name, lists in KW.items():
    describe_lists(name, lists)
    g = next(iter(lists))
    print(f"  e.g. {g}: {', '.join(lists[g]['keywords'])}")

## Scoring

Per utterance, after normalisation:
- **WER** from a jiwer alignment.
- **B-WER and U-WER** ([Le et al., 2021](https://arxiv.org/abs/2104.02194)): errors split by whether the reference word (for substitutions and deletions) or the inserted word is on the group's list. B-WER is divided by the number of listed words in the references, and U-WER by all other reference words.
- **Target recall:** the share of target-term occurrences in the reference that also appear in the transcript (bag-of-words per utterance).
- **Distractor insertions:** distractor words in the transcript. No one in the group said them, so each one is a hallucination prompted by the list. Reported per 100 utterances.

In [ ]:
def score_utt(ref, hyp, kw):
    r, h = norm(ref).split(), norm(hyp).split()
    listed = kw["targets"] | kw["distractors"]
    c = Counter(n=len(r), b_n=sum(w in listed for w in r))
    c["u_n"] = c["n"] - c["b_n"]
    if h:
        out = jiwer.process_words(" ".join(r), " ".join(h))
        for ch in out.alignments[0]:
            if ch.type == "equal":
                continue
            if ch.type in ("substitute", "delete"):
                for w in r[ch.ref_start_idx:ch.ref_end_idx]:
                    c["b_err" if w in listed else "u_err"] += 1
                    c["err"] += 1
            if ch.type == "insert":
                for w in h[ch.hyp_start_idx:ch.hyp_end_idx]:
                    c["b_err" if w in listed else "u_err"] += 1
                    c["err"] += 1
    else:
        c["err"], c["b_err"], c["u_err"] = c["n"], c["b_n"], c["u_n"]
    rc, hc = Counter(r), Counter(h)
    c["tgt_ref"] = sum(rc[t] for t in kw["targets"])
    c["tgt_hit"] = sum(min(rc[t], hc[t]) for t in kw["targets"])
    c["dis_ins"] = sum(hc[t] for t in kw["distractors"])
    c["utts"] = 1
    return c


COUNT_COLS = ["n", "err", "b_n", "b_err", "u_n", "u_err", "tgt_ref", "tgt_hit", "dis_ins", "utts"]


def rates(s):
    return {"WER": s["err"] / s["n"], "B-WER": s["b_err"] / max(s["b_n"], 1), "U-WER": s["u_err"] / s["u_n"],
            "target recall": s["tgt_hit"] / max(s["tgt_ref"], 1), "distractor ins / 100 utts": 100 * s["dis_ins"] / s["utts"]}


kw_demo = dict(targets={"fexofenadine", "eczema"}, distractors={"naproxen"})
print(score_utt("I'd suggest fexofenadine for the eczema.", "I'd suggest fexo fenadine and naproxen for the eczema.", kw_demo))

## Models

All three models are loaded one at a time and freed afterwards, so the T4's 16 GB holds whichever is running. Each `transcribe(audios, keywords)` takes a batch of 16 kHz arrays that share a keyword list (`None` for conditions without one) and returns transcripts.

### Baseline: Whisper-small (conditions A and B)

Whisper's standard way to bias toward vocabulary is a text prompt in its decoder context (`prompt_ids`). The decoder holds 448 tokens in total, so a list that would push the prompt past that is shortened, and the notebook reports how often that happens.

In [ ]:
def free():
    """Release GPU memory after the caller has deleted its model and transcribe references."""
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


FP = torch.float16 if DEVICE == "cuda" else torch.float32
WHISPER_TRUNCATED = Counter()


def load_whisper():
    proc = WhisperProcessor.from_pretrained(WHISPER_ID, revision=WHISPER_REV)
    model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, revision=WHISPER_REV, dtype=FP).to(DEVICE).eval()

    def prompt_ids(keywords):
        kws = list(keywords)
        while True:
            ids = proc.get_prompt_ids(WHISPER_KW_PROMPT.format(keywords=", ".join(kws)), return_tensors="pt")
            if len(ids) <= WHISPER_MAX_PROMPT or len(kws) <= 1:
                break
            kws.pop()
        WHISPER_TRUNCATED["lists"] += 1
        WHISPER_TRUNCATED["shortened"] += len(kws) < len(keywords)
        return ids.to(DEVICE)

    def transcribe(audios, keywords):
        feats = proc.feature_extractor(audios, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
        extra = {"prompt_ids": prompt_ids(keywords)} if keywords else {}
        with torch.inference_mode():
            out = model.generate(feats.input_features.to(DEVICE, FP), attention_mask=feats.attention_mask.to(DEVICE),
                                 language="en", task="transcribe", max_new_tokens=MAX_NEW_TOKENS, **extra)
        return [t.strip() for t in proc.batch_decode(out, skip_special_tokens=True)]

    return model, transcribe

### New approach: Granite Speech 5.0 (conditions C, D and E)

The 2B model follows the model card: a chat prompt with an `<|audio|>` placeholder, greedy decoding, and left padding for batches. Condition E only appends `Keywords: …` to condition D's prompt. Turbo CTC decodes non-autoregressively and takes no prompt, so it has no keyword condition.

In [ ]:
def granite_dtype():
    if GRANITE_DTYPE != "auto":
        return getattr(torch, GRANITE_DTYPE)
    if DEVICE == "cuda":
        return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # a T4 has no native bfloat16
    return torch.float32


def load_granite():
    proc = AutoProcessor.from_pretrained(GRANITE_ID, revision=GRANITE_REV)
    tok = proc.tokenizer
    tok.padding_side = "left"
    model = AutoModelForSpeechSeq2Seq.from_pretrained(GRANITE_ID, revision=GRANITE_REV, device_map=DEVICE,
                                                      dtype=granite_dtype()).eval()

    def prompt(keywords):
        q = GRANITE_KW_PROMPT.format(keywords=", ".join(keywords)) if keywords else GRANITE_PROMPT
        return tok.apply_chat_template([{"role": "user", "content": "<|audio|>" + q}], tokenize=False, add_generation_prompt=True)

    def transcribe(audios, keywords):
        p = prompt(keywords)
        inputs = proc([p] * len(audios), [torch.from_numpy(a) for a in audios], device=DEVICE, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1)
        new = out[:, inputs["input_ids"].shape[1]:]
        return [t.strip() for t in tok.batch_decode(new, add_special_tokens=False, skip_special_tokens=True)]

    print("Granite 2B dtype:", model.dtype)
    print("prompt D:", repr(prompt(None)))
    return model, transcribe


def load_ctc():
    proc = AutoProcessor.from_pretrained(CTC_ID, revision=CTC_REV)
    model = AutoModelForCTC.from_pretrained(CTC_ID, revision=CTC_REV, device_map=DEVICE).eval()

    def transcribe(audios, keywords):
        inputs = proc(audios, sampling_rate=proc.feature_extractor.sampling_rate, device=model.device)
        inputs.to(model.device, dtype=model.dtype)
        with torch.inference_mode():
            out = model.generate(**inputs)
        return [t.strip() for t in proc.batch_decode(out, skip_special_tokens=True)]

    return model, transcribe


LOADERS = {"whisper": load_whisper, "granite": load_granite, "ctc": load_ctc}

## Evaluation

Each condition runs over each dataset in batches of utterances from the same group, sorted by length. Transcripts and batch timings are appended to a JSONL file per condition and dataset, so a rerun skips finished utterances. RTFx is seconds of audio per second of processing; it includes feature extraction and decoding, but not audio loading.

Before the full run, a sanity check transcribes 8 PriMock57 utterances with every model. If Granite's float16 output is empty or garbled on a T4, set `GRANITE_DTYPE = "float32"` in Config.

In [ ]:
def load_audio(row):
    if row.dataset == "primock57":
        y, sr = sf.read(row.audio, start=int(row.start * 16000), stop=int(row.end * 16000), dtype="float32")
    else:
        y, sr = librosa.load(row.audio, sr=16000, mono=True)   # AfriSpeech is recorded at 44.1 kHz
    if y.ndim > 1:
        y = y.mean(axis=1)
    return np.ascontiguousarray(y, dtype=np.float32)


def cache_path(name, cond):
    return CACHE_DIR / f"{name}_{cond}.jsonl"


def read_cache(name, cond):
    p = cache_path(name, cond)
    return pd.DataFrame([json.loads(l) for l in p.read_text().splitlines()]) if p.exists() else pd.DataFrame()


def run_condition(cond, name, df, transcribe, bs):
    _, gets_kw, label = CONDITIONS[cond]
    done = set(read_cache(name, cond).get("utt_id", []))
    todo = df[~df.utt_id.isin(done)]
    if todo.empty:
        return
    with open(cache_path(name, cond), "a") as f, tqdm(total=len(todo), desc=f"{label} | {name}") as bar:
        for g, sub in todo.groupby("group", sort=False):
            kws = KW[name][g]["keywords"] if gets_kw else None
            sub = sub.sort_values("duration")
            for i in range(0, len(sub), bs):
                batch = sub.iloc[i:i + bs]
                audios = [load_audio(r) for r in batch.itertuples()]
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                hyps = transcribe(audios, kws)
                if DEVICE == "cuda":
                    torch.cuda.synchronize()
                dt = time.perf_counter() - t0
                total = batch.duration.sum()
                for r, hyp in zip(batch.itertuples(), hyps):
                    f.write(json.dumps({"utt_id": r.utt_id, "hyp": hyp, "proc_sec": dt * r.duration / total}) + "\n")
                f.flush()
                bar.update(len(batch))

In [ ]:
check = primock.sample(8, random_state=SEED)
for model_name in ("whisper", "ctc", "granite"):
    model, transcribe = LOADERS[model_name]()
    hyps = transcribe([load_audio(r) for r in check.itertuples()], None)
    wer = jiwer.wer([norm(t) for t in check.ref], [norm(h) or "<empty>" for h in hyps])
    print(f"\n{model_name}: WER on 8 utterances {wer:.1%}")
    for ref, hyp in list(zip(check.ref, hyps))[:3]:
        print(f"  ref: {ref}\n  hyp: {hyp}")
    del model, transcribe
    free()

In [ ]:
PEAK_GB = {}
peak_file = CACHE_DIR / "peak_gb.json"
if peak_file.exists():
    PEAK_GB = json.loads(peak_file.read_text())

for model_name in ("whisper", "ctc", "granite"):
    conds = [c for c, (m, _, _) in CONDITIONS.items() if m == model_name]
    if all(set(df.utt_id) <= set(read_cache(name, c).get("utt_id", [])) for c in conds for name, df in DATA.items()):
        print(f"{model_name}: all cached")
        continue
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model, transcribe = LOADERS[model_name]()
    for name, df in DATA.items():
        for c in conds:
            run_condition(c, name, df, transcribe, BATCH[model_name])
    if DEVICE == "cuda":
        PEAK_GB[model_name] = torch.cuda.max_memory_allocated() / 1e9
        peak_file.write_text(json.dumps(PEAK_GB))
    del model, transcribe
    free()

if WHISPER_TRUNCATED["lists"]:
    print(f"Whisper prompt shortened for {WHISPER_TRUNCATED['shortened']} of {WHISPER_TRUNCATED['lists']} batches")

### Scores per utterance and per condition

Confidence intervals resample whole clusters (a PriMock57 consultation, or an AfriSpeech speaker) 2,000 times, because utterances from the same conversation or voice aren't independent. Paired differences resample the same clusters for both conditions.

In [ ]:
rows = []
for name, df in DATA.items():
    meta = df.set_index("utt_id")
    for c in CONDITIONS:
        res = read_cache(name, c)
        if res.empty:
            continue
        for r in res.itertuples():
            m = meta.loc[r.utt_id]
            s = score_utt(m.ref, r.hyp, KW[name][m.group])
            rows.append({"dataset": name, "condition": c, "utt_id": r.utt_id, "group": m.group, "cluster": m.cluster,
                         "subset": m.subset, "duration": m.duration, "proc_sec": r.proc_sec, "ref": m.ref, "hyp": r.hyp,
                         **{k: s[k] for k in COUNT_COLS}})
results = pd.DataFrame(rows)
results.to_csv("results.csv", index=False)


def boot_ci(df, metric, n=N_BOOT, seed=SEED):
    per = df.groupby("cluster")[COUNT_COLS].sum()
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(per), size=(n, len(per)))
    arr = per.to_numpy()
    vals = [rates(dict(zip(COUNT_COLS, arr[i].sum(0))))[metric] for i in idx]
    return np.percentile(vals, [2.5, 97.5])


def boot_diff(df_a, df_b, metric, n=N_BOOT, seed=SEED):
    a = df_a.groupby("cluster")[COUNT_COLS].sum()
    b = df_b.groupby("cluster")[COUNT_COLS].sum().reindex(a.index)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    A, B = a.to_numpy(), b.to_numpy()
    d = [rates(dict(zip(COUNT_COLS, A[i].sum(0))))[metric] - rates(dict(zip(COUNT_COLS, B[i].sum(0))))[metric] for i in idx]
    point = rates(dict(zip(COUNT_COLS, A.sum(0))))[metric] - rates(dict(zip(COUNT_COLS, B.sum(0))))[metric]
    lo, hi = np.percentile(d, [2.5, 97.5])
    return point, lo, hi


METRICS = ["WER", "B-WER", "U-WER", "target recall", "distractor ins / 100 utts"]
summary = []
for (name, c), g in results.groupby(["dataset", "condition"]):
    model_name, _, label = CONDITIONS[c]
    row = {"dataset": name, "condition": label, "utterances": len(g), "hours": g.duration.sum() / 3600}
    for m in METRICS:
        row[m] = rates(g[COUNT_COLS].sum())[m]
        lo, hi = boot_ci(g, m)
        row[f"{m} 95% CI"] = f"{lo:.3g}–{hi:.3g}"
    row["RTFx"] = g.duration.sum() / g.proc_sec.sum()
    row["peak GPU GB"] = PEAK_GB.get(model_name)
    summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv("summary.csv", index=False)
summary[["dataset", "condition", "utterances", "WER", "B-WER", "U-WER", "target recall", "distractor ins / 100 utts", "RTFx", "peak GPU GB"]].round(3)

In [ ]:
diffs = []
for name in results.dataset.unique():
    for x, y, what in COMPARISONS:
        gx = results[(results.dataset == name) & (results.condition == x)]
        gy = results[(results.dataset == name) & (results.condition == y)]
        if gx.empty or gy.empty:
            continue
        for m in METRICS:
            point, lo, hi = boot_diff(gx, gy, m)
            diffs.append({"dataset": name, "comparison": f"{x} − {y} ({what})", "metric": m, "difference": point,
                          "95% CI": f"{lo:.3g} to {hi:.3g}", "CI excludes 0": lo > 0 or hi < 0})
diffs = pd.DataFrame(diffs)
diffs.to_csv("differences.csv", index=False)
diffs[diffs.metric.isin(["WER", "B-WER", "target recall", "distractor ins / 100 utts"])].round(4)

## Results

In [ ]:
order = [CONDITIONS[c][2] for c in CONDITIONS]
fig, axes = plt.subplots(1, len(DATA), figsize=(6.5 * len(DATA), 4.2), squeeze=False)
for ax, name in zip(axes[0], DATA):
    s = summary[summary.dataset == name].set_index("condition").reindex(order)
    x = np.arange(len(s))
    for j, (m, color) in enumerate([("WER", "#4C72B0"), ("B-WER", "#DD8452"), ("U-WER", "#55A868")]):
        err = np.array([[v - float(ci.split("–")[0]), float(ci.split("–")[1]) - v] if isinstance(ci, str) else [0, 0]
                        for v, ci in zip(s[m], s[f"{m} 95% CI"])]).T
        ax.bar(x + (j - 1) * 0.27, s[m], 0.27, yerr=err, capsize=2, label=m, color=color)
    ax.set_xticks(x, [c.split(" ", 1)[0] for c in s.index])
    ax.set_title(name)
    ax.set_ylabel("error rate (lower is better)")
    ax.legend(frameon=False)
fig.suptitle("WER overall, on listed words (B-WER) and on all other words (U-WER)")
fig.tight_layout()
fig.savefig(FIG_DIR / "wer.png", dpi=150)
plt.show()
print("\n".join(order))

In [ ]:
fig, axes = plt.subplots(1, len(DATA), figsize=(6.5 * len(DATA), 4.2), squeeze=False)
for ax, name in zip(axes[0], DATA):
    for _, r in summary[summary.dataset == name].iterrows():
        xy = (r["distractor ins / 100 utts"], r["target recall"])
        ax.scatter(*xy, s=60)
        ax.annotate(r["condition"].split(" ", 1)[0], xy, xytext=(5, 3), textcoords="offset points")
    ax.set_xlabel("distractor insertions per 100 utterances (lower is better)")
    ax.set_ylabel("target-term recall (higher is better)")
    ax.set_title(name)
fig.suptitle("What keyword lists buy (recall) and cost (hallucinated distractors)")
fig.tight_layout()
fig.savefig(FIG_DIR / "recall_vs_insertions.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for name, marker in zip(DATA, "os"):
    s = summary[summary.dataset == name]
    ax.scatter(s.RTFx, s.WER, marker=marker, s=60, label=name)
    for r in s.itertuples():
        ax.annotate(r.condition.split(" ", 1)[0], (r.RTFx, r.WER), xytext=(5, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set_xlabel("RTFx on a T4 (seconds of audio per second, log scale)")
ax.set_ylabel("WER")
ax.legend(frameon=False)
ax.set_title("Accuracy vs speed")
fig.tight_layout()
fig.savefig(FIG_DIR / "speed.png", dpi=150)
plt.show()

In [ ]:
by_subset = []
for (name, c, sub), g in results.groupby(["dataset", "condition", "subset"]):
    by_subset.append({"dataset": name, "condition": c, "subset": sub, "utterances": len(g), **rates(g[COUNT_COLS].sum())})
by_subset = pd.DataFrame(by_subset)
by_subset.to_csv("by_subset.csv", index=False)

display(by_subset[by_subset.dataset == "primock57"].pivot(index="subset", columns="condition", values="WER").round(3))

if "afrispeech" in DATA:
    acc = by_subset[(by_subset.dataset == "afrispeech") & (by_subset.utterances >= MIN_ACCENT_CLIPS)]
    piv = acc.pivot(index="subset", columns="condition", values="WER").sort_values("A")
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(piv) + 1.5))
    for c, marker in zip(["A", "C", "D", "E"], "o^sD"):
        if c in piv:
            ax.scatter(piv[c], range(len(piv)), marker=marker, label=CONDITIONS[c][2], s=30)
    ax.set_yticks(range(len(piv)), piv.index)
    ax.set_xlabel("WER")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title(f"AfriSpeech clinical: WER by accent ({MIN_ACCENT_CLIPS} clips each)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "afrispeech_accents.png", dpi=150)
    plt.show()

## Qualitative examples

Five utterances where the conditions disagree on a target term: where keywords fixed one, where Granite with keywords (E) wrote a distractor nobody said, and where every condition missed.

In [ ]:
wide = results.pivot_table(index=["dataset", "utt_id"], columns="condition", values=["hyp", "tgt_hit", "dis_ins"], aggfunc="first")
# kept separate from `wide`, whose columns are two-level (value, condition)
meta = results.drop_duplicates(["dataset", "utt_id"]).set_index(["dataset", "utt_id"])[["ref", "tgt_ref"]].reindex(wide.index)


def show(title, mask, k=1):
    picks = wide[mask].sample(min(k, int(mask.sum())), random_state=SEED) if mask.any() else wide.iloc[:0]
    for idx, r in picks.iterrows():
        print(f"\n### {title}  [{idx[0]} {idx[1]}]\nREF: {meta.at[idx, 'ref']}")
        for c in CONDITIONS:
            if ("hyp", c) in r.index:
                print(f"  {c}: {r[('hyp', c)]}")
    if picks.empty:
        print(f"\n### {title}: no example")


has = lambda c: ("tgt_hit", c) in wide.columns
if has("E") and has("D"):
    show("Keywords fixed a target term (E over D)", (wide[("tgt_hit", "E")] > wide[("tgt_hit", "D")]), k=2)
    show("Granite with keywords wrote a distractor (E)", wide[("dis_ins", "E")] > 0, k=2)
if has("B"):
    show("Whisper with keywords wrote a distractor (B)", wide[("dis_ins", "B")] > 0)
show("Every condition missed a target term", (meta["tgt_ref"] > 0) & np.logical_and.reduce([wide[("tgt_hit", c)] == 0 for c in CONDITIONS if has(c)]))

## Interpretation

TODO(you): after running, write up:
- Did keyword lists help Granite (E vs D) and Whisper (B vs A) on target terms, and did they hurt the other words (U-WER)?
- How many distractors did each keyword condition write per 100 utterances, and would that be acceptable in a clinical note?
- Doctor vs patient channel on PriMock57, and which AfriSpeech accents were hardest for each model.
- Is Turbo CTC's speed worth its accuracy here?
- When would you use each model?

## Limitations and next steps

- **Oracle keyword lists.** Each list contains the group's real rare terms, as in most contextual-biasing studies. A real medication list would miss some terms and include more that are never said; the distractors only approximate the second half.
- **Rare words are not all medical.** The Zipf threshold also catches names, typos in the reference (PriMock57 has *athsma*) and unusual everyday words. Check the printed lists.
- **Reference quality.** PriMock57 transcripts are utterance-level and include `<UNSURE>` passages, so some "errors" are reference errors. AfriSpeech references are prompt sentences, so a speaker's deviation from the prompt counts as a model error. Its domain labels are noisy too: some "clinical" clips are legal or general text.
- **Accent means two or three speakers.** In the AfriSpeech test set, the median accent with 30+ clinical clips has only 2–3 speakers, so per-accent WER mostly reflects those voices, not the accent. Read the per-accent chart as speaker variation, and rely on the overall numbers, whose intervals resample speakers.
- **Untuned prompts and one sample size.** Whisper's prompt format and Granite's keyword prompt are used as documented, without tuning. AfriSpeech uses 30 clips per accent, so per-accent intervals are wide.
- **float16 on a T4.** Granite is published in bfloat16; on a T4 it runs in float16 unless `GRANITE_DTYPE = "float32"`.
- Next: realistic lists (for example, terms from the clinician's PriMock57 note), list length vs insertion rate, and whisper-large-v3-turbo as a stronger baseline.

## Environment

In [ ]:
print({m: r for m, r in [(WHISPER_ID, WHISPER_REV), (GRANITE_ID, GRANITE_REV), (CTC_ID, CTC_REV)]})
print("PriMock57 commit:", PRIMOCK_SHA)
!pip freeze | grep -iE "^(transformers|torch|torchaudio|jiwer|wordfreq|huggingface-hub|tokenizers|librosa|soundfile|pandas|numpy)=="

In [ ]:
import zipfile
from datetime import datetime

zip_path = Path(f"granite_kwb_results_{datetime.now():%Y%m%d_%H%M}.zip")
files = [*FIG_DIR.glob("*.png"), *(Path(f) for f in ["results.csv", "summary.csv", "differences.csv", "by_subset.csv"])]
files += [*CACHE_DIR.glob("*.jsonl"), *CACHE_DIR.glob("*.json")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if f.exists():
            z.write(f, f.relative_to(BASE_DIR) if BASE_DIR in f.parents else f)
print(f"{zip_path}: {len(z.namelist())} files, {zip_path.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("Not on Colab: the zip is next to the notebook.")